In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, concatenate_datasets,Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold
from collections import Counter

c:\Users\eesha\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CEFR_LEVELS = ["A1", "A2", "B2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}

In [3]:
# Load all English datasets
english_datasets = [
    load_dataset("UniversalCEFR/readme_en")["train"],
    load_dataset("UniversalCEFR/cefr_asag_en")["train"],
    load_dataset("UniversalCEFR/icle500_en")["train"],
    load_dataset("UniversalCEFR/cefr_sp_en")["train"],
    load_dataset("UniversalCEFR/elg_cefr_en")["train"],
    load_dataset("UniversalCEFR/cambridge_exams_en")["train"],
]
english_data = concatenate_datasets(english_datasets)

c:\Users\eesha\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eesha\.cache\huggingface\hub\datasets--UniversalCEFR--readme_en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 2822/2822 [00:00<00:00, 42116.16 ex

In [4]:
english_data

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
    num_rows: 14663
})

In [5]:
# Filter to keep only valid CEFR levels
filtered_data = english_data.filter(lambda x: x["cefr_level"] in CEFR_LEVELS)

Filter:   0%|          | 0/14663 [00:00<?, ? examples/s]

Filter: 100%|██████████| 14663/14663 [00:00<00:00, 54079.01 examples/s]


In [6]:
# Remove duplicate texts
df = filtered_data.to_pandas().drop_duplicates(subset="text", keep="first")
filtered_data = Dataset.from_pandas(df)

In [7]:
filtered_data

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text', '__index_level_0__'],
    num_rows: 7058
})

In [8]:
df["cefr_level"].value_counts()

cefr_level
B2    4572
A2    2138
A1     348
Name: count, dtype: int64

In [9]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

c:\Users\eesha\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eesha\.cache\huggingface\hub\models--EuroBERT--EuroBERT-210m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed

In [10]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [11]:
# Tokenize the dataset
tokenized_data = filtered_data.map(preprocess, batched=True, remove_columns=filtered_data.column_names)

# Split English into train/val
n = len(tokenized_data)
train_end = int(0.8 * n)
dev_end = int(0.9 * n)

ds_train = tokenized_data.select(range(0, train_end))
ds_dev   = tokenized_data.select(range(train_end, dev_end))
ds_test  = tokenized_data.select(range(dev_end, n))

Map: 100%|██████████| 7058/7058 [00:00<00:00, 8566.78 examples/s] 


In [12]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS), trust_remote_code=True)

A new version of the following files was downloaded from https://huggingface.co/EuroBERT/EuroBERT-210m:
- configuration_eurobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/EuroBERT/EuroBERT-210m:
- modeling_eurobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'den

In [13]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [14]:
# Training args
args = TrainingArguments(
    output_dir="./eurobert_cefr_english_only",  
    num_train_epochs=3, 
    per_device_train_batch_size=2,              
    per_device_eval_batch_size=3,                
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_weighted_f1",
    greater_is_better=True,
    seed=42,
    learning_rate=3.6e-5,
    warmup_ratio=0.1,
    gradient_accumulation_steps=16,      
    optim="adamw_torch_fused",                   
    lr_scheduler_type="linear",                  
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    save_total_limit=1,
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  
)

C:\Users\c24082331\AppData\Local\Temp\ipykernel_30220\3133241838.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
# Train on English-only data
trainer.train()

# Evaluate on Dev set
trainer.evaluate()

# Save model, tokenizer, and trainer state
save_dir = "./eurobert_cefr_english_only/final_model"
trainer.save_model(save_dir)                    
tokenizer.save_pretrained(save_dir)            
trainer.state.save_to_json(os.path.join(save_dir, "trainer_state.json"))  

print(f"Model saved to {save_dir}")

Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B2 Precision,B2 Recall,B2 F1
1,0.514300,0.081653,0.983003,0.983160,0.986527,0.983003,1.000000,0.250000,0.400000,0.500000,0.727273,0.592593,0.994194,0.991317,0.992754
2,0.311900,0.088987,0.961756,0.969924,0.982569,0.961756,0.500000,0.250000,0.333333,0.250000,0.727273,0.372093,0.997024,0.969609,0.983125
3,0.169500,0.123638,0.966006,0.971101,0.980377,0.966006,0.000000,0.000000,0.000000,0.290323,0.818182,0.428571,0.997037,0.973951,0.985359


c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Model saved to ./eurobert_cefr_english_only/final_model
